# Weed Management Chatbot

This notebook builds a retrieval-augmented question-answering system for the `NoxWeedMangementGuide2019LasAnimas.pdf` dataset.

The goal is to let a farmer ask questions about weeds found in a field and receive answers grounded in the PDF. The app should help with weed identification, lifecycle details, growth stage information, and control recommendations.


## RAG Pipeline

The application follows this retrieval-augmented generation flow:

1. Load the weed management PDF.
2. Extract text page by page with PyMuPDF.
3. Clean spacing and line-break issues from extracted text.
4. Split the document into weed-entry chunks.
5. Attach metadata to each chunk.
6. Create embeddings with OpenAI `text-embedding-3-small`.
7. Store chunk text, embeddings, and metadata in ChromaDB.
8. Accept a farmer's question.
9. Embed the question.
10. Retrieve the most relevant chunks from ChromaDB.
11. Send the retrieved context and question to an OpenAI chat model.
12. Return a grounded answer with source information.


## 1. **Read the PDF** with PyMuPDF.
   - The weed management guide has structured pages and herbicide tables.
   - PyMuPDF is a good first choice because it handles page layout better than basic text extraction tools.

In [1]:
from pathlib import Path
import json
import re
import fitz  # PyMuPDF


PDF_PATH = Path("data/NoxWeedMangementGuide2019LasAnimas.pdf")

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

RAW_TEXT_PATH = RAW_DIR / "extracted_raw_column_blocks.txt"
CLEAN_TEXT_PATH = PROCESSED_DIR / "extracted_clean_column_blocks.txt"
PAGES_JSON_PATH = PROCESSED_DIR / "pdf_pages_column_blocks.json"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def clean_pdf_text(text: str) -> str:
    # Force all line breaks to use Unix format: \n
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Fix words split across lines, like "under-\n\nstand" -> "understand"
    text = re.sub(r"(\w)-\s*\n+\s*(\w)", r"\1\2", text)

    # Remove extra spaces but keep line breaks
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]

    # Collapse too many blank lines
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_page_by_column_blocks(page, num_columns=4):
    # Split page into columns, then sort text blocks top-to-bottom inside each column
    page_width = page.rect.width
    all_blocks = page.get_text("blocks")

    column_texts = []

    for column_index in range(num_columns):
        x0 = column_index * page_width / num_columns
        x1 = (column_index + 1) * page_width / num_columns

        column_blocks = []

        for block in all_blocks:
            bx0, by0, bx1, by1, text, block_no, block_type = block

            # Keep only text blocks
            if block_type != 0 or not text.strip():
                continue

            # Use block center to decide which column it belongs to
            block_center_x = (bx0 + bx1) / 2

            if x0 <= block_center_x < x1:
                column_blocks.append(block)

        # Sort within the column from top to bottom
        column_blocks = sorted(column_blocks, key=lambda block: block[1])

        column_text = "\n\n".join(block[4].strip() for block in column_blocks)

        if column_text:
            column_texts.append(
                f"--- COLUMN {column_index + 1} ---\n\n{column_text}"
            )

    return "\n\n".join(column_texts)


pages = []
raw_parts = []
clean_parts = []

with fitz.open(PDF_PATH) as doc:
    for page_num, page in enumerate(doc, start=1):
        raw_text = extract_page_by_column_blocks(page)
        clean_text = clean_pdf_text(raw_text)

        pages.append(
            {
                "source": PDF_PATH.name,
                "page": page_num,
                "raw_text": raw_text,
                "clean_text": clean_text,
            }
        )

        raw_parts.append(f"--- PAGE {page_num} ---\n\n{raw_text}")
        clean_parts.append(f"--- PAGE {page_num} ---\n\n{clean_text}")


RAW_TEXT_PATH.write_text("\n\n".join(raw_parts), encoding="utf-8")
CLEAN_TEXT_PATH.write_text("\n\n".join(clean_parts), encoding="utf-8")

PAGES_JSON_PATH.write_text(
    json.dumps(pages, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Extracted {len(pages)} pages")
print(f"Saved raw text: {RAW_TEXT_PATH}")
print(f"Saved clean text: {CLEAN_TEXT_PATH}")
print(f"Saved page JSON: {PAGES_JSON_PATH}")

Extracted 6 pages
Saved raw text: data/raw/extracted_raw_column_blocks.txt
Saved clean text: data/processed/extracted_clean_column_blocks.txt
Saved page JSON: data/processed/pdf_pages_column_blocks.json


In [2]:
print("--- Raw Text Snippet ---")
print(pages[0]["raw_text"][:1000])

--- Raw Text Snippet ---
--- COLUMN 1 ---

Weed 
Management 
Pocket Guide

Colorado 
Stale 
University, 
U.S. 
Department 
of 
Agriculture 
and 
Colorado 
Counties 
cooperating. 
Cooperative Extension programs are available to all 
without 
discrimination. 
To 
simplify 
technical 
terminology, trade names of products will be used. No 
endorsement of products named is intended nor 
criticism implied of products not motioned.

--- COLUMN 2 ---

This brochure was created to increase 
awareness of Noxious Weeds, the 
importance of identification, the 
importance of a weed management 
program, and some methods of weed 
control based on local, state and 
national research-based information.

How do I control weeds on

my property?

1. Identify the weeds on your property.

2. Once a weed is identified, under-

stand the life cycle of the weed 
• winter or summer annual  
• biennial  
• simple or creeping perennial 
3. Understand the types of controls

• Preventative 
● Biological 
• Cultural

In [3]:
print("--- Cleaned Text Snippet ---")
print(pages[0]["clean_text"][:1000])

--- Cleaned Text Snippet ---
--- COLUMN 1 ---

Weed
Management
Pocket Guide

Colorado
Stale
University,
U.S.
Department
of
Agriculture
and
Colorado
Counties
cooperating.
Cooperative Extension programs are available to all
without
discrimination.
To
simplify
technical
terminology, trade names of products will be used. No
endorsement of products named is intended nor
criticism implied of products not motioned.

--- COLUMN 2 ---

This brochure was created to increase
awareness of Noxious Weeds, the
importance of identification, the
importance of a weed management
program, and some methods of weed
control based on local, state and
national research-based information.

How do I control weeds on

my property?

1. Identify the weeds on your property.

2. Once a weed is identified, understand the life cycle of the weed
• winter or summer annual
• biennial
• simple or creeping perennial
3. Understand the types of controls

• Preventative
● Biological
• Cultural

● Chemical
• Mechanical

4. Deve

#### Observation:

The PDF extraction is now done page by page, and each page is split into 4 vertical columns. For each column, text blocks are read top to bottom, then the extractor moves left to right across the page. This matches the brochure-style layout much better.

raw_text keeps the original PyMuPDF extraction with extra spaces, broken line words, and original block text.

clean_text keeps the same page/column structure but lightly cleans spacing, line breaks, and split words, so it is better for chunking and RAG.

In [4]:
import json
from pathlib import Path


pages = json.loads(
    Path("data/processed/pdf_pages_column_blocks.json").read_text(encoding="utf-8")
)

page = pages[0]

print("RAW TEXT:")
print(page["raw_text"][650:1050])

print("\n" + "=" * 50 + "\n")

print("CLEAN TEXT:")
print(page["clean_text"][650:1050])

RAW TEXT:
-based information.

How do I control weeds on

my property?

1. Identify the weeds on your property.

2. Once a weed is identified, under-

stand the life cycle of the weed 
• winter or summer annual  
• biennial  
• simple or creeping perennial 
3. Understand the types of controls

• Preventative 
● Biological 
• Cultural 
 
● Chemical 
• Mechanical

4. Develop a weed management plan

• planning


CLEAN TEXT:
 I control weeds on

my property?

1. Identify the weeds on your property.

2. Once a weed is identified, understand the life cycle of the weed
• winter or summer annual
• biennial
• simple or creeping perennial
3. Understand the types of controls

• Preventative
● Biological
• Cultural

● Chemical
• Mechanical

4. Develop a weed management plan

• planning saves money and

increases effectiveness


#### Observation:  Raw Text vs Clean Text

`raw_text` keeps the PDF extraction exactly as PyMuPDF returned it, including broken words, extra spaces, and noisy blank lines. `clean_text` keeps the same page and column structure, but fixes those issues so the text is better for chunking and RAG.

clean_text is now fine for RAG because it removes extraction noise while preserving useful line breaks and section structure.

Examples:

RAW:
under-

stand

CLEAN:
understand

## 2. **Chunk the extracted text** using document structure-based chunking.
https://www.pinecone.io/learn/chunking-strategies/

Use structure-based chunking by page column, because the PDF is a 4-column brochure and each column contains one coherent topic. Most columns are weed entries, while the first page contains general weed-management information. Store simple metadata: source, page, column, and topic.

Preserve metadata for source page and column number. 
```
{
    "source": "NoxWeedMangementGuide2019LasAnimas.pdf",
    "page": 2,
    "column": 1,
    "topic": "Bull thistle"
}
```

In [5]:
import json
import re
from pathlib import Path


PAGES_JSON_PATH = Path("data/processed/pdf_pages_column_blocks.json")
CHUNKS_JSON_PATH = Path("data/processed/chunks.json")


def extract_topic(column_text: str) -> str:
    # Use the first meaningful line as the topic
    lines = [
        line.strip()
        for line in column_text.splitlines()
        if line.strip() and not line.startswith("--- COLUMN")
    ]

    if not lines:
        return "Unknown"

    return lines[0]


def split_page_into_column_chunks(page: dict) -> list[dict]:
    # Split page text into column sections
    parts = re.split(r"--- COLUMN (\d+) ---", page["clean_text"])

    chunks = []

    # parts looks like: ["", "1", "text...", "2", "text...", ...]
    for i in range(1, len(parts), 2):
        column_number = int(parts[i])
        column_text = parts[i + 1].strip()

        if not column_text:
            continue

        topic = extract_topic(column_text)

        chunks.append(
            {
                "id": f"page_{page['page']}_column_{column_number}",
                "text": column_text,
                "metadata": {
                    "source": page["source"],
                    "page": page["page"],
                    "column": column_number,
                    "topic": topic,
                },
            }
        )

    return chunks


pages = json.loads(PAGES_JSON_PATH.read_text(encoding="utf-8"))

chunks = []

for page in pages:
    chunks.extend(split_page_into_column_chunks(page))


CHUNKS_JSON_PATH.write_text(
    json.dumps(chunks, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Created {len(chunks)} chunks")
print(f"Saved chunks to: {CHUNKS_JSON_PATH}")



Created 24 chunks
Saved chunks to: data/processed/chunks.json


In [8]:
print("\nExample chunk:")
print(json.dumps(chunks[4], indent=2, ensure_ascii=False)[:1500])


Example chunk:
{
  "id": "page_2_column_1",
  "text": "Bull thistle\nCirsium vulgare (Savi) Tenore\n\nKeys to Id\n\n•Leaves are pricklyhairy above and cottony below.\n•Heads cobwebbypubescent\n•Flowers are composite and purple\n\nIdentification\n•\nLifecycle: Biennial\n•\nGrowth form: Forb/herb\n•\nFlower: Flowers are 1.5-2 in wide and clustered\nat the ends of branches. The flower bracts are\nsomewhat tapered and covered with spines\n(Whitson et al. 1996).\n•\nSeeds/Fruit: Seeds are capped with a circle of\nplume-like white hairs.\n•\nLeaves: Leaves are alternate. Bull are the only\nthistles in Colorado that are prickly hairy on the\ntop surface of the leaves. They are cottony-hairy\non the undersides.\n•\nStems: In mature plants the leaves extend down,\nclasping the stem and are divided into segments\n(i.e. strongly decurrent).\n\nControl\n•\nMech: sever the root below the soil surface\n\n•\nBio: Urophora stylata, a fly predator, can be\nused to help control this thistle.\n\nHERBICI

In [9]:
for chunk in chunks:
    print(chunk["id"], "->", chunk["metadata"]["topic"])

page_1_column_1 -> Weed
page_1_column_2 -> This brochure was created to increase
page_1_column_3 -> What are noxious weeds?
page_1_column_4 -> Weed Control Methods
page_2_column_1 -> Bull thistle
page_2_column_2 -> Canada thistle
page_2_column_3 -> Musk thistle
page_2_column_4 -> Scotch thistle
page_3_column_1 -> Diffuse knapweed
page_3_column_2 -> Spotted knapweed
page_3_column_3 -> Russian knapweed
page_3_column_4 -> Houndstongue
page_4_column_1 -> Common Mullein
page_4_column_2 -> Locoweed (Wooly)
page_4_column_3 -> Salt Cedar (Tamarisk)
page_4_column_4 -> Hoary Cress (Whitetop)
page_5_column_1 -> Kochia
page_5_column_2 -> Russian Olive
page_5_column_3 -> Perennial Pepperweed
page_5_column_4 -> Field Bindweed
page_6_column_1 -> Common Cocklebur
page_6_column_2 -> Rabbit Brush
page_6_column_3 -> Winter Annuals
page_6_column_4 -> Backyard Weed Control Tips


#### Observation: 
Chunking looks good to proceed because each page-column chunk maps cleanly to one coherent topic, with weed entries like Bull thistle, Canada thistle, and Musk thistle correctly separated, while general sections like Weed Control Methods and Backyard Weed Control Tips are also kept as their own chunks.

## 3. **Create embeddings** with OpenAI `text-embedding-3-small`.
   - Each chunk will be converted into a vector representation for semantic search.

Using text-embedding-3-small for this project.
Why:
- dataset is tiny: only 24 chunks.
- The text is English and domain-specific but not huge.
- text-embedding-3-small is much cheaper: 0.02 per 1M tokens vs text-embedding-3-large at 0.13 per 1M tokens.
- OpenAI’s docs list text-embedding-3-small as 1536 dimensions and text-embedding-3-large as 3072 dimensions, so large gives more capacity, but we probably do not need it here.

In [10]:
import json
import os
from pathlib import Path

import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv

# Load OPENAI_API_KEY from .env
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")

# Paths
CHUNKS_JSON_PATH = Path("data/processed/chunks.json")
CHROMA_DB_DIR = Path("data/chroma_db")

# Load chunks
chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding="utf-8"))

# Create persistent Chroma client
client = chromadb.PersistentClient(path=str(CHROMA_DB_DIR))

# Use OpenAI embedding model
embedding_function = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small",
)


# Create or load collection
collection = client.get_or_create_collection(
    name="weed_management_guide",
    embedding_function=embedding_function,
)

# Prepare data for Chroma
ids = [chunk["id"] for chunk in chunks]
documents = [chunk["text"] for chunk in chunks]
metadatas = [chunk["metadata"] for chunk in chunks]


## 4. **Store documents, embeddings, and metadata** in ChromaDB.
   - ChromaDB is a good fit for this demo because it stores the text chunk, embedding, and metadata together.
   - FAISS can also be used for vector search, but metadata usually has to be stored separately.

## ChromaDB Metadata Design

Each stored chunk should include metadata that helps the app cite sources and filter results

{
    "source": "NoxWeedMangementGuide2019LasAnimas.pdf",
    "page": 2,
    "column": 3,
    "topic": "Musk thistle"
}


In [11]:
# Add chunks to Chroma; upsert is safe to rerun
collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)


print(f"Stored {collection.count()} chunks in ChromaDB")
print(f"Chroma DB saved at: {CHROMA_DB_DIR}")

Stored 24 chunks in ChromaDB
Chroma DB saved at: data/chroma_db


In [12]:
results = collection.query(
    query_texts=["When should herbicide be applied to musk thistle?"],
    n_results=3,
)

for doc, metadata in zip(results["documents"][0], results["metadatas"][0]):
    print(metadata)
    print(doc[:500])
    print("-" * 80)

{'topic': 'Musk thistle', 'page': 2, 'source': 'NoxWeedMangementGuide2019LasAnimas.pdf', 'column': 3}
Musk thistle
Carduus nulans

Keys to Id

•Broad, spine-tipped
bracts located under the flower
•Flowering heads are terminal, solitary, and
usually nodding
•Grows up to 6 feet tall

Identification
•
Lifecycle: Biennial, or sometimes winter annual
•
Growth form: Forb
•
Flower: Heads are terminal, solitary, 1 1/2-3 in
wide, and usually nodding. Deep rose, violet or
purple, occasionally white. Flowers are subtended
by broad, spine-tipped bracts. May-July.
•
Seeds/Fruit: One-seeded oblong fruit (ache
--------------------------------------------------------------------------------
{'topic': 'Bull thistle', 'page': 2, 'source': 'NoxWeedMangementGuide2019LasAnimas.pdf', 'column': 1}
Bull thistle
Cirsium vulgare (Savi) Tenore

Keys to Id

•Leaves are pricklyhairy above and cottony below.
•Heads cobwebbypubescent
•Flowers are composite and purple

Identification
•
Lifecycle: Biennial
•
Growth fo

#### Observation: 
Embedding and vector storage are working correctly; all 24 chunks were stored in ChromaDB, and the retrieval test returned the relevant Musk thistle chunk first.

## 5. **Answer questions using retrieved PDF context**

The chatbot will:
1. Accept a farmer's question.
2. Retrieve the most relevant chunks from ChromaDB.
3. Build a context string from those chunks.
4. Send the question and context to an OpenAI chat model.
5. Return an answer grounded only in the PDF context.

In [14]:
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [15]:
def retrieve_documents(question: str, n_results: int = 3):
    # Retrieve relevant chunks from ChromaDB
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
    )

    documents = []

    for text, metadata in zip(results["documents"][0], results["metadatas"][0]):
        documents.append(
            Document(
                page_content=text,
                metadata=metadata,
            )
        )

    return documents

In [16]:
# Test retrieval before adding the LLM
test_docs = retrieve_documents(
    "When should herbicide be applied to musk thistle?",
    n_results=3,
)

for doc in test_docs:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("-" * 80)

{'page': 2, 'column': 3, 'topic': 'Musk thistle', 'source': 'NoxWeedMangementGuide2019LasAnimas.pdf'}
Musk thistle
Carduus nulans

Keys to Id

•Broad, spine-tipped
bracts located under the flower
•Flowering heads are terminal, solitary, and
usually nodding
•Grows up to 6 feet tall

Identification
•
Lifecycle: Biennial, or sometimes winter annual
•
Growth form: Forb
•
Flower: Heads are terminal, solitary, 1 1/2-3 in
wide, and usually nodding. Deep rose, violet or
purple, occasionally white. Flowers are subtended
by broad, spine-tipped bracts. May-July.
•
Seeds/Fruit: One-seeded oblong fruit (ache
--------------------------------------------------------------------------------
{'source': 'NoxWeedMangementGuide2019LasAnimas.pdf', 'column': 1, 'page': 2, 'topic': 'Bull thistle'}
Bull thistle
Cirsium vulgare (Savi) Tenore

Keys to Id

•Leaves are pricklyhairy above and cottony below.
•Heads cobwebbypubescent
•Flowers are composite and purple

Identification
•
Lifecycle: Biennial
•
Growth fo

In [17]:
def create_context(documents):
    # Combine retrieved documents into one context block
    return "\n\n---\n\n".join(
        f"Source: {doc.metadata['source']}, "
        f"Page: {doc.metadata['page']}, "
        f"Column: {doc.metadata['column']}, "
        f"Topic: {doc.metadata['topic']}\n\n"
        f"{doc.page_content}"
        for doc in documents
    )

In [24]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_template(
    """
You are a weed management assistant for farmers.

Answer the question using only the provided PDF context.

Important rules:
- If the context directly answers the question, answer clearly and include source citations.
- If the context does not directly answer the question, say:
  "The PDF does not provide enough information to answer this question."
- Do not cite sources when the context is not relevant to the question.
- Do not guess or use outside knowledge.

Include practical details when available:
- identification clues
- lifecycle
- growth form
- mechanical control
- biological control
- chemical control
- herbicide timing

Context:
{context}

Question:
{question}
"""
)

In [19]:
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.2,
    api_key=os.getenv("OPENAI_API_KEY"),
)

runnable parallel is to run 2 components in parallel and runnable passthrough is to keep the question still available throughtout the chain

In [25]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


rag_chain = (
    RunnableParallel(
        {
            "question": RunnablePassthrough(),
            "context": RunnableLambda(retrieve_documents) | RunnableLambda(create_context),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [26]:
answer = rag_chain.invoke(
    "When should herbicide be applied to musk thistle?"
)

print(answer)

Herbicide should be applied to musk thistle during the following times:

- Aminopyralid (Milestone): Apply in spring from rosette to early bolting stage or in fall to rosettes.
- Metsulfuron (Escort XP): Apply in spring from rosette through very early flower stage.
- Chlorsulfuron (Telar): Apply in spring from rosette through early flower stage.

(Source: NoxWeedMangementGuide2019LasAnimas.pdf, Page 2, Column 3)


In [27]:
test_questions = [
    "I have a weed with purple flower heads, usually nodding, broad spine-tipped bracts, and a rosette present last fall. Which thistle is this?",
    "What is the correct timing window for treating diffuse knapweed at the rosette stage?",
    "How should saltcedar be controlled if mowing is ineffective?",
    "What herbicide should I use for crabgrass in my lawn?",
]


for question in test_questions:
    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    answer = rag_chain.invoke(question)
    print(answer)

    print("\n")

QUESTION:
I have a weed with purple flower heads, usually nodding, broad spine-tipped bracts, and a rosette present last fall. Which thistle is this?

ANSWER:
Based on your description of a weed with purple flower heads that are usually nodding, broad spine-tipped bracts, and a rosette present last fall, the thistle you have is **Musk thistle (Carduus nutans)**.

Key identification clues from the PDF:
- Flower heads are terminal, solitary, and usually nodding.
- Flowers are deep rose, violet, or purple.
- Broad, spine-tipped bracts under the flower.
- Lifecycle: Biennial or sometimes winter annual, with a basal rosette well developed in the fall.
- Rosette present last fall fits the biennial lifecycle and rosette stage described.

(Source: NoxWeedMangementGuide2019LasAnimas.pdf, Page 2, Column 3, Topic: Musk thistle)


QUESTION:
What is the correct timing window for treating diffuse knapweed at the rosette stage?

ANSWER:
The correct timing window for treating diffuse knapweed at the r

## 6. **Evaluate answer quality** with Braintrust.
   - Evaluation should check retrieval quality, correctness, groundedness, completeness, and source citation quality.


## Evaluation Plan

Use Braintrust to evaluate the final chatbot.

Evaluation should measure:

- **Retrieval relevance**: Did ChromaDB return the correct weed entry or control section?
- **Answer correctness**: Does the answer match the PDF?
- **Groundedness**: Does the model avoid unsupported claims?
- **Completeness**: Does the answer include identification details, lifecycle, control method, or timing when relevant?
- **Source traceability**: Does the answer include useful page/source references?

Suggested evaluation questions:

- “I have a weed with Purple flower heads, nodding, spine-tipped bracts, rosette 
present last fall. Which thistle is this?” 
-  “What is the correct timing window for treating diAuse knapweed at the rosette 
stage?” 
- “How should saltcedar be controlled if mowing is ineAective?”
- Which thistle has purple flower heads that are usually nodding and broad spine-tipped bracts?
- What is the lifecycle of bull thistle?
- How should Canada thistle be controlled mechanically?
- When should herbicide be applied to musk thistle?
- What are the main categories of weed control methods?
- What should the app say if the PDF does not contain enough information?
